In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("omnichannel_demand_inventory_data.csv")

# 2. Create a temporary in-memory SQLite database
conn = sqlite3.connect(":memory:")

# 3. Push the Pandas DataFrame into a SQL table named 'sales'
df.to_sql("sales", conn, index=False, if_exists="replace")

print("Data successfully loaded into SQLite!")

Data successfully loaded into SQLite!


# Phase 1: Demand Forecasting 

In [24]:
forecast_query = """
WITH DateRange AS (
    SELECT DISTINCT Date FROM sales
),
SKUs AS (
    SELECT DISTINCT SKU, Category FROM sales
),
Calendar AS (
    SELECT d.Date, s.SKU, s.Category
    FROM DateRange d
    CROSS JOIN SKUs s
),
DailySales AS (
    SELECT 
        c.Date,
        c.SKU,
        c.Category,
        COALESCE(SUM(sa.Units_Sold), 0) AS Actual_Units,
        ROUND(COALESCE(SUM(sa.Total_Revenue), 0), 2) AS Total_Revenue,
        ROUND(COALESCE(SUM(sa.Total_Cost), 0), 2) AS Total_Cost,
        ROUND(COALESCE(SUM(sa.Gross_Margin), 0), 2) AS Gross_Margin
    FROM Calendar c
    LEFT JOIN sales sa 
        ON sa.Date = c.Date AND sa.SKU = c.SKU
    GROUP BY c.Date, c.SKU, c.Category
),
RollingForecast AS (
    SELECT 
        Date,
        SKU,
        Category,
        Actual_Units,
        Total_Revenue,
        Total_Cost,
        Gross_Margin,
        ROUND(AVG(Actual_Units) OVER (
            PARTITION BY SKU 
            ORDER BY Date 
            ROWS BETWEEN 90 PRECEDING AND 1 PRECEDING
        ), 2) AS Forecast_Units_Raw,
        COUNT(Actual_Units) OVER (
            PARTITION BY SKU 
            ORDER BY Date 
            ROWS BETWEEN 90 PRECEDING AND 1 PRECEDING
        ) AS Historical_Days_Count
    FROM DailySales
)
SELECT 
    Date,
    SKU,
    Category,
    Actual_Units,
    Total_Revenue,
    Total_Cost,
    Gross_Margin,
    CASE 
        WHEN Historical_Days_Count >= 30 THEN Forecast_Units_Raw
        ELSE NULL 
    END AS Forecast_Units,
    Historical_Days_Count,
    CASE 
        WHEN Historical_Days_Count >= 30 THEN ABS(Actual_Units - Forecast_Units_Raw)
        ELSE NULL 
    END AS Absolute_Error
FROM RollingForecast;
"""
 
forecast_df = pd.read_sql_query(forecast_query, conn)
 
forecast_df.to_csv("fact_daily_sales_forecast.csv", index=False)
print("fact_daily_sales_forecast.csv exported successfully!")

fact_daily_sales_forecast.csv exported successfully!


# Phase 2 Inventory Health, Saftey Stock, and ABC Analysis

In [ ]:
inventory_query = """
WITH SKU_Metrics AS (
    SELECT 
        SKU,
        Category,
        AVG(Lead_Time_Days) AS Avg_Lead_Time,
        SUM(Total_Revenue) AS Total_SKU_Revenue,
        AVG(On_Hand_Inventory) AS Current_Avg_Inventory
    FROM sales
    GROUP BY SKU, Category
),
ABC_Prepped AS (
    SELECT 
        *,
        SUM(Total_SKU_Revenue) OVER () AS Total_Company_Revenue,
        -- Fix: explicit ROWS frame + SKU tiebreaker, so SKUs with identical
        -- revenue don't get merged into one cumulative "peer group" value
        SUM(Total_SKU_Revenue) OVER (
            ORDER BY Total_SKU_Revenue DESC, SKU
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS Cumulative_Revenue
    FROM SKU_Metrics
)
SELECT 
    SKU,
    Category,
    Avg_Lead_Time,
    Current_Avg_Inventory,
    ROUND(Total_SKU_Revenue, 2) AS Total_Revenue,
    ROUND((Cumulative_Revenue / Total_Company_Revenue) * 100, 2) AS Cumulative_Revenue_Pct,
    CASE 
        WHEN (Cumulative_Revenue / Total_Company_Revenue) <= 0.80 THEN 'A-Class (High Value)'
        WHEN (Cumulative_Revenue / Total_Company_Revenue) <= 0.95 THEN 'B-Class (Moderate Value)'
        ELSE 'C-Class (Low Value)'
    END AS ABC_Class
FROM ABC_Prepped
ORDER BY Total_SKU_Revenue DESC;
"""
 
inventory_df = pd.read_sql_query(inventory_query, conn)
 
 
# =========================================================================
# Build a dense Date x SKU demand calendar (zero-filled) in Pandas
# =========================================================================
# for days with actual sales. Grouping directly on `df` -- as the original
# code did -- computes mean/std over selling days only, understating true
# demand variability (and, for a plain AVG, overstating average demand) by
# ignoring every zero-sale day entirely.
all_dates = df['Date'].unique()
sku_lookup = df[['SKU', 'Category']].drop_duplicates()
 
calendar = pd.MultiIndex.from_product(
    [all_dates, sku_lookup['SKU']], names=['Date', 'SKU']
).to_frame(index=False)
calendar = calendar.merge(sku_lookup, on='SKU', how='left')
 
daily_actuals = df.groupby(['Date', 'SKU'], as_index=False)['Units_Sold'].sum()
 
calendar = calendar.merge(daily_actuals, on=['Date', 'SKU'], how='left')
calendar['Units_Sold'] = calendar['Units_Sold'].fillna(0)
 
# True daily demand mean & std, now correctly including zero-sale days
demand_stats = (
    calendar.groupby('SKU')['Units_Sold']
    .agg(Avg_Daily_Demand='mean', Demand_StdDev='std')
    .reset_index()
)
 
# Fix: SKUs with only one day of history ever will have Demand_StdDev = NaN
# (sample std needs >= 2 points). Fall back to the SKU's Category-level
# average std as a reasonable estimate, rather than silently propagating NaN
# into Safety_Stock / Reorder_Point_ROP downstream.
demand_stats = demand_stats.merge(sku_lookup, on='SKU', how='left')
category_avg_std = (
    demand_stats.groupby('Category')['Demand_StdDev'].transform('mean')
)
n_missing_std = demand_stats['Demand_StdDev'].isna().sum()
demand_stats['Demand_StdDev'] = demand_stats['Demand_StdDev'].fillna(category_avg_std)
demand_stats = demand_stats.drop(columns='Category')
 
if n_missing_std:
    print(f"Note: {n_missing_std} SKU(s) had insufficient history for a standalone "
          f"StdDev and were backfilled with their Category's average StdDev.")
 
 
# =========================================================================
# Merge demand stats into inventory dataframe
# =========================================================================
inventory_df = inventory_df.merge(demand_stats, on='SKU', how='left')
 
 
# =========================================================================
# Supply Chain Formulas
# =========================================================================
# 1. Safety Stock = Z-Score (1.65 for 95% service level) * Demand_StdDev * sqrt(Avg_Lead_Time)
#    Note: this assumes lead time is effectively constant per SKU. If lead
#    times vary meaningfully between orders, consider incorporating lead-time
#    variance too:
#    SS = Z * sqrt(Avg_Lead_Time * Demand_StdDev**2 + Avg_Daily_Demand**2 * LeadTime_StdDev**2)
inventory_df['Safety_Stock'] = (
    1.65 * inventory_df['Demand_StdDev'] * (inventory_df['Avg_Lead_Time'] ** 0.5)
).round(0)
 
# 2. Reorder Point (ROP) = (Avg Daily Demand * Avg Lead Time) + Safety Stock
inventory_df['Reorder_Point_ROP'] = (
    (inventory_df['Avg_Daily_Demand'] * inventory_df['Avg_Lead_Time']) + inventory_df['Safety_Stock']
).round(0)
 
print(inventory_df[['SKU', 'Category', 'ABC_Class', 'Avg_Daily_Demand',
                     'Demand_StdDev', 'Safety_Stock', 'Reorder_Point_ROP']].head())

        SKU     Category             ABC_Class  Avg_Daily_Demand  \
0  SKU_H302  Home Office  A-Class (High Value)         26.151847   
1  SKU_A205   Appliances  A-Class (High Value)         25.956224   
2  SKU_E102  Electronics  A-Class (High Value)         26.143639   
3  SKU_H305  Home Office  A-Class (High Value)         26.251710   
4  SKU_A201   Appliances  A-Class (High Value)         26.019152   

   Demand_StdDev  Safety_Stock  Reorder_Point_ROP  
0       6.850272          34.0              269.0  
1       7.113425          50.0              517.0  
2       6.974505          43.0              409.0  
3       6.993508          35.0              271.0  
4       7.183098          47.0              463.0  


# Phase 3 Promotional and Bundle Lift Analysis

In [11]:
import numpy as np

In [22]:
promo_query = """
SELECT 
    Category,
    Is_Promotion,
    Is_Bundle,
    COUNT(DISTINCT Date) AS Total_Days,
    SUM(Units_Sold) AS Total_Units_Sold,
    ROUND(AVG(Units_Sold), 1) AS Avg_Daily_Units,
    ROUND(SUM(Total_Revenue), 2) AS Total_Revenue,
    ROUND(AVG(Total_Revenue), 2) AS Avg_Daily_Revenue,
    ROUND(SUM(Gross_Margin), 2) AS Total_Gross_Margin,
    ROUND(AVG(Gross_Margin), 2) AS Avg_Daily_Margin,
    ROUND((SUM(Gross_Margin) / NULLIF(SUM(Total_Revenue), 0)) * 100, 2) AS Gross_Margin_Pct
FROM sales
GROUP BY Category, Is_Promotion, Is_Bundle
ORDER BY Category, Is_Promotion DESC, Is_Bundle DESC;
"""
promo_df = pd.read_sql_query(promo_query, conn)
 

# Calculate Lift % Metrics in Pandas (Comparing Promo vs. Non-Promo Baseline)
def safe_pct_change(new, base):
    """
    Percentage change from base -> new, guarding against:
      - base == 0 (division by zero)
      - base < 0 (a '% lift' over a negative baseline is misleading, since
        the sign of the result flips in a way that doesn't mean 'improvement')
    Returns np.nan in either edge case, with the caller free to inspect why.
    """
    if base == 0 or pd.isna(base):
        return np.nan
    if base < 0:
        return np.nan
    return ((new - base) / base) * 100
 
 
def calculate_lift(df):
    results = []
 
    for cat in df['Category'].unique():
        cat_data = df[df['Category'] == cat]
 
        # Baseline: Non-Promo & Non-Bundle
        baseline = cat_data[(cat_data['Is_Promotion'] == 0) & (cat_data['Is_Bundle'] == 0)]
        if baseline.empty:
            continue
        if len(baseline) > 1:
            # Group-by keys should make this impossible; if it happens, the
            # data has duplicate/inconsistent category keys worth investigating
            # (e.g. mismatched casing or whitespace in Category).
            raise ValueError(
                f"Expected exactly one baseline row for category '{cat}', "
                f"found {len(baseline)}. Check for duplicate/inconsistent category values."
            )
 
        base_units = baseline['Avg_Daily_Units'].iloc[0]
        base_margin = baseline['Avg_Daily_Margin'].iloc[0]
 
        # --- Standard Promo Lift: Promo, but not Bundle ---
        promo = cat_data[(cat_data['Is_Promotion'] == 1) & (cat_data['Is_Bundle'] == 0)]
        if not promo.empty:
            promo_units = promo['Avg_Daily_Units'].iloc[0]
            promo_margin = promo['Avg_Daily_Margin'].iloc[0]
 
            results.append({
                'Category': cat,
                'Type': 'Standard Promo',
                'Baseline_Daily_Units': base_units,
                'Promo_Daily_Units': promo_units,
                'Volume_Lift_Pct': round(safe_pct_change(promo_units, base_units), 2),
                'Baseline_Daily_Margin': base_margin,
                'Promo_Daily_Margin': promo_margin,
                'Margin_Lift_Pct': round(safe_pct_change(promo_margin, base_margin), 2),
            })
 
        # --- Bundle Lift: bundles are always promotional in this dataset,
        # so Is_Promotion == 1 & Is_Bundle == 1 correctly captures all bundles.
        bundle = cat_data[(cat_data['Is_Promotion'] == 1) & (cat_data['Is_Bundle'] == 1)]
        if not bundle.empty:
            if len(bundle) > 1:
                raise ValueError(
                    f"Expected exactly one bundle row for category '{cat}', "
                    f"found {len(bundle)}. Check for duplicate/inconsistent category values."
                )
            bundle_units = bundle['Avg_Daily_Units'].iloc[0]
            bundle_margin = bundle['Avg_Daily_Margin'].iloc[0]
 
            results.append({
                'Category': cat,
                'Type': 'Bundle Promo',
                'Baseline_Daily_Units': base_units,
                'Promo_Daily_Units': bundle_units,
                'Volume_Lift_Pct': round(safe_pct_change(bundle_units, base_units), 2),
                'Baseline_Daily_Margin': base_margin,
                'Promo_Daily_Margin': bundle_margin,
                'Margin_Lift_Pct': round(safe_pct_change(bundle_margin, base_margin), 2),
            })
 
    return pd.DataFrame(results)
 
 
lift_summary_df = calculate_lift(promo_df)
print("=== PROMOTIONAL & BUNDLE LIFT SUMMARY ===")
print(lift_summary_df.to_string(index=False))

=== PROMOTIONAL & BUNDLE LIFT SUMMARY ===
   Category           Type  Baseline_Daily_Units  Promo_Daily_Units  Volume_Lift_Pct  Baseline_Daily_Margin  Promo_Daily_Margin  Margin_Lift_Pct
Accessories Standard Promo                  12.4               15.9            28.23                 471.72              313.83           -33.47
Accessories   Bundle Promo                  12.4               18.9            52.42                 471.72              381.59           -19.11
 Appliances Standard Promo                  12.2               15.6            27.87                 574.94              310.94           -45.92
 Appliances   Bundle Promo                  12.2               18.6            52.46                 574.94              366.86           -36.19
Electronics Standard Promo                  12.3               15.5            26.02                 656.55              496.51           -24.38
Electronics   Bundle Promo                  12.3               17.9            45.53    

# Export dataframes as three csv files 

In [23]:
# 1. Export Primary Fact Table (Daily Sales + Rolling Forecast)
forecast_df.to_csv("fact_daily_sales_forecast.csv", index=False)

# 2. Export Inventory Dimension Table (Safety Stock, ROP, ABC Tiers)
inventory_df.to_csv("dim_sku_inventory_health.csv", index=False)

# 3. Export Promotional Performance Summary Table
promo_df.to_csv("fact_promotional_performance.csv", index=False)

print("All 3 clean CSV files exported successfully for Tableau/Power BI!")

All 3 clean CSV files exported successfully for Tableau/Power BI!


In [25]:
fact_totals = forecast_df.groupby('SKU')['Total_Revenue'].sum().round(2)
dim_totals = inventory_df.set_index('SKU')['Total_Revenue']
 
revenue_comparison = fact_totals.to_frame('Fact_Total').join(dim_totals.rename('Dim_Total'))
revenue_comparison['Diff'] = (revenue_comparison['Fact_Total'] - revenue_comparison['Dim_Total']).round(2)
 
mismatches = revenue_comparison[revenue_comparison['Diff'] != 0]
if mismatches.empty:
    print("\nTie-out check passed: fact and dim revenue totals match for all SKUs.")
else:
    print("\nTie-out check FAILED for the following SKUs:")
    print(mismatches)



Tie-out check passed: fact and dim revenue totals match for all SKUs.
